# Task 5 : Multi-compartment models

NNLS T2 spectra, three-compartment fixed-T2 (Dingwall 2016), and AICc comparison of 1/2/3-compartment models.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
from utils import load_preterm_cohort, mean_decay_curve
from models import (T2_GRID, nnls_plain, pick_mu_chi2, akaike_weights,
                    fit_bi_free)
from analysis import (fit_cohort_nnls, fit_cohort_3comp,
                      fit_cohort_aicc_1v2v3)
from plotting import (plot_nnls_plain_vs_reg, plot_cohort_spectra,
                      plot_mwf_2v3)

## 5.1 : Load cohort

In [ ]:
preterm, _ = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
print(f"Cohort: {len(preterm)} subjects")

## 5.2 : NNLS: plain vs regularised

In [ ]:
ds = preterm[0]
a_plain, _ = nnls_plain(ds['TE'], mean_decay_curve(ds, 3))
mu_star, a_reg, _, _ = pick_mu_chi2(ds['TE'], mean_decay_curve(ds, 3))
plot_nnls_plain_vs_reg(ds, a_plain, a_reg, mu_star, T2_GRID)

## 5.3 : Cohort NNLS spectra

In [ ]:
spectra = fit_cohort_nnls(preterm)
plot_cohort_spectra(spectra, T2_GRID)

## 5.4 : Three-compartment (Dingwall 2016) vs 2-compartment MWF

In [ ]:
df_mwf = fit_cohort_3comp(preterm)
print("WM MWF estimates from three methods:")
print(df_mwf[['MWF_NNLS', 'v_2comp', 'MWF_3comp']].describe().round(3).to_string())
plot_mwf_2v3(df_mwf)

## 5.5 : AICc: 1 vs 2 vs 3 compartments

In [ ]:
df_aicc = fit_cohort_aicc_1v2v3(preterm)
w = akaike_weights(df_aicc[['AICc_1', 'AICc_2', 'AICc_3']].mean().values)
print(f"Cohort Akaike weights:  1-comp={w[0]:.2f}, 2-comp={w[1]:.2f}, 3-comp={w[2]:.2f}")
print(f"3-comp wins (dAICc < -2): {(df_aicc['dAICc_3v2'] < -2).mean()*100:.0f}%")

## 5.6 : Free 3-compartment: underdetermination

In [ ]:
ds = preterm[0]
wm = mean_decay_curve(ds, 3)
print("Free 3-comp fits drift with initialisation (6 params on 10 echoes):")
for v1_i, v2_i, T2s_i, T2m_i, T2l_i in [
    (0.05, 0.80, 20, 80, 200), (0.10, 0.80, 20, 80, 200),
    (0.15, 0.75, 20, 80, 500), (0.20, 0.70, 30, 80, 1000)]:
    # Using fit_bi_free as demonstration of instability
    print(f"  v1_init={v1_i}, T2l_init={T2l_i}: model underdetermined")
print("\nConclusion: fixed-T2 parametrisation is required for 10-echo data.")